In [1]:
# ==========================================
# CELDA 1: Importación de Librerías
# ==========================================
import numpy as np
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
import warnings

# Ocultar advertencias para una vista limpia en la presentación
warnings.filterwarnings('ignore')

In [2]:
# ==========================================
# CELDA 2: Motor Matemático (Backend)
# ==========================================
class WindkesselSimulator:
    def __init__(self, R=1.0, C=1.2, HR=75):
        """
        R: Resistencia Periférica (mmHg*s/mL)
        C: Compliance Arterial (mL/mmHg)
        HR: Frecuencia Cardíaca (latidos/min)
        """
        self.R = R
        self.C = C
        self.HR = HR

    def flujo_aortico(self, t):
        """Simula el flujo de sangre desde el corazón en sístole y diástole"""
        T_ciclo = 60.0 / self.HR
        T_sistole = T_ciclo * 0.3

        t_local = t % T_ciclo

        if t_local <= T_sistole:
            return 400 * np.sin(np.pi * t_local / T_sistole)
        else:
            return 0.0

    def edo_presion(self, t, P):
        """Ecuación Diferencial Ordinaria: dP/dt = Q/C - P/(R*C)"""
        Q = self.flujo_aortico(t)
        dP_dt = (Q / self.C) - (P / (self.R * self.C))
        return dP_dt

    def simular(self):
        """Ejecuta la simulación y estabiliza el modelo"""
        tiempo_total = 15.0
        t_span = (0, tiempo_total)
        t_eval = np.linspace(0, tiempo_total, 3000)
        P_inicial = [80.0]

        # Solver de Runge-Kutta de orden 4(5)
        sol = solve_ivp(self.edo_presion, t_span, P_inicial, t_eval=t_eval, method='RK45')

        # Recortar los últimos 5 segundos para estado estable
        idx_start = np.searchsorted(sol.t, tiempo_total - 5.0)
        t_plot = sol.t[idx_start:]
        p_plot = sol.y[0][idx_start:]

        q_plot = np.array([self.flujo_aortico(ti) for ti in t_plot])
        t_plot = t_plot - t_plot[0]

        return t_plot, q_plot, p_plot

In [3]:
# ==========================================
# CELDA 3: Lógica de Gráficos y Eventos
# ==========================================
out_grafica = widgets.Output()

slider_r = widgets.FloatSlider(value=1.0, min=0.4, max=3.0, step=0.1, description='Resistencia (R):', style={'description_width': 'initial'})
slider_c = widgets.FloatSlider(value=1.2, min=0.3, max=3.0, step=0.1, description='Compliance (C):', style={'description_width': 'initial'})
slider_hr = widgets.IntSlider(value=75, min=40, max=180, step=1, description='Frec. Cardiaca:', style={'description_width': 'initial'})

btn_normal = widgets.Button(description="Normal", button_style='success')
btn_hiper = widgets.Button(description="Hipertensión", button_style='danger')
btn_hipo = widgets.Button(description="Hipotensión", button_style='info')
btn_arte = widgets.Button(description="Arteriosclerosis", button_style='warning')

def actualizar_grafica(*args):
    """Renderiza las gráficas cada vez que un parámetro cambia"""
    with out_grafica:
        clear_output(wait=True)

        sim = WindkesselSimulator(R=slider_r.value, C=slider_c.value, HR=slider_hr.value)
        tiempo, flujo, presion = sim.simular()

        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 7), sharex=True)

        # Gráfica Superior: Flujo Aórtico
        ax1.plot(tiempo, flujo, color='royalblue', linewidth=2)
        ax1.set_title('Simulador Cardiovascular - Modelo Windkessel', fontsize=14, fontweight='bold')
        ax1.set_ylabel('Flujo Aórtico (mL/s)', fontsize=12)
        ax1.grid(True, linestyle='--', alpha=0.7)
        ax1.fill_between(tiempo, flujo, color='royalblue', alpha=0.1)

        # Gráfica Inferior: Presión Arterial
        ax2.plot(tiempo, presion, color='crimson', linewidth=2)
        ax2.set_xlabel('Tiempo (Segundos)', fontsize=12)
        ax2.set_ylabel('Presión Arterial (mmHg)', fontsize=12)
        ax2.grid(True, linestyle='--', alpha=0.7)

        # Cálculo de Presión Sistólica y Diastólica
        p_sistolica = np.max(presion)
        p_diastolica = np.min(presion)

        # Dibujar líneas guía
        ax2.axhline(p_sistolica, color='gray', linestyle=':', linewidth=1.5)
        ax2.axhline(p_diastolica, color='gray', linestyle=':', linewidth=1.5)

        # Etiquetas numéricas en la gráfica
        ax2.text(0.1, p_sistolica + 2, f'Sistólica: {p_sistolica:.1f} mmHg', color='darkred', fontweight='bold')
        ax2.text(0.1, p_diastolica - 8, f'Diastólica: {p_diastolica:.1f} mmHg', color='darkred', fontweight='bold')

        plt.tight_layout()
        plt.show()

# Lógica de los botones para mutar el modelo a patologías
def set_normal(b): slider_r.value=1.0; slider_c.value=1.2; slider_hr.value=75
def set_hiper(b): slider_r.value=1.8; slider_c.value=1.0; slider_hr.value=85
def set_hipo(b): slider_r.value=0.6; slider_c.value=1.4; slider_hr.value=60
def set_arte(b): slider_r.value=1.1; slider_c.value=0.4; slider_hr.value=80

btn_normal.on_click(set_normal)
btn_hiper.on_click(set_hiper)
btn_hipo.on_click(set_hipo)
btn_arte.on_click(set_arte)

slider_r.observe(actualizar_grafica, names='value')
slider_c.observe(actualizar_grafica, names='value')
slider_hr.observe(actualizar_grafica, names='value')

In [4]:
# ==========================================
# CELDA 4: Creación de la Interfaz y Despliegue
# ==========================================
controles = widgets.VBox([
    widgets.HTML("<b>Parámetros Fisiológicos (Modificar manualmente):</b>"),
    slider_r, slider_c, slider_hr
])

botones = widgets.VBox([
    widgets.HTML("<b>Simular Patologías (Pre-configuraciones):</b>"),
    widgets.HBox([btn_normal, btn_hiper]),
    widgets.HBox([btn_hipo, btn_arte])
])

# Ensamblaje de las dos columnas de controles
panel_superior = widgets.HBox([controles, botones], layout=widgets.Layout(justify_content='space-around', margin='20px 0px'))

# Inicializar y mostrar
actualizar_grafica()
display(panel_superior, out_grafica)

Output()